<a href="https://colab.research.google.com/github/ambre-bissi/ESG-Portfolio-Optimizer/blob/main/Toolkit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sustainable Quant Investing Toolkit

ESG Portfolio Risk Analyzer (Projet personnel GitHub)

• Développement d'un outil Python d'analyse quantitative et ESG de portefeuilles financiers.

• Construction d'indicateurs de risque (volatilité, Sharpe ratio, Value-at-Risk).

• Création et comparaison de portefeuilles optimisés selon des critères financiers et de durabilité.

• Analyse de la qualité des données ESG, détection d'anomalies et visualisation des résultats.

In [9]:
import yfinance as yf
import pandas as pd
import numpy as np

# -----------------------------
# 1. Univers d'actions
# -----------------------------

tickers = [
    "MC.PA",    # LVMH
    "AIR.PA",   # Airbus
    "DG.PA",    # Vinci
    "OR.PA",    # L'Oréal
    "SAN.PA"    # Sanofi
]

# -----------------------------
# 2. Téléchargement des données
# -----------------------------

prices = yf.download(
    tickers,
    start="2021-01-01",
    end="2026-01-01",
    auto_adjust=True
)["Close"]

# -----------------------------
# 3. Rendements journaliers
# -----------------------------

returns = prices.pct_change().dropna()

# -----------------------------
# 4. Indicateurs quantitatifs
# -----------------------------

annual_return = returns.mean() * 252
annual_volatility = returns.std() * np.sqrt(252)

# -----------------------------
# 5. Scores ESG (fictifs)
# -----------------------------

esg_scores = pd.Series({
    "MC.PA": 72,
    "AIR.PA": 81,
    "DG.PA": 78,
    "OR.PA": 85,
    "SAN.PA": 88
})

[*********************100%***********************]  5 of 5 completed


In [2]:
# -----------------------------
# 6. Tableau récapitulatif
# -----------------------------

summary = pd.DataFrame({
    "Annual Return": annual_return,
    "Volatility": annual_volatility,
    "ESG Score": esg_scores
})

print(summary.round(3))

        Annual Return  Volatility  ESG Score
AIR.PA          0.210       0.286         81
DG.PA           0.132       0.211         78
MC.PA           0.105       0.291         72
OR.PA           0.074       0.235         85
SAN.PA          0.074       0.224         88


An error has occurred. The ticker symbol might be wrong or you might need to wait to continue.
None


ne fonctionne pas, on reste avec les scores fictifs

In [ ]:
# -----------------------------
# 5. Scores ESG
# -----------------------------
#!pip install yesg
import json
import yesg

tickers_code = [
    "MC",    # LVMH
    "AIR",   # Airbus
    "DG",    # Vinci
    "OR",    # L'Oréal
    "SAN",    # Sanofi
    "AAPL"    #apple
]

print(yesg.get_historic_esg("AAPL"))

import requests

# Read in your symbols
#sp_500 = pd.read_csv("SP_500_tickers.csv")

# Endpoint
url = "https://query2.finance.yahoo.com/v1/finance/esgChart"

# List of dataframes
dataframes = []

for symbol in tickers_code:
    response = requests.get(url, params={"symbol": symbol})
    if response.ok:
        df = pd.DataFrame(response.json()["esgChart"]["result"][0]["symbolSeries"])
        df["symbol"] = symbol
        dataframes.append(df)
    else:
        print(f"Error fetching data for {symbol}")

# Concatenate all dataframes

df = pd.concat(dataframes)
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")
df.set_index("timestamp", inplace=True)
df.sort_index(inplace=True)
df.head()

# Création du portefeuille

In [3]:
# Poids du portefeuille

weights = np.array([0.2, 0.2, 0.2, 0.2, 0.2])

# Rendement attendu

portfolio_return = np.dot(weights, annual_return)

# Matrice de covariance

cov_matrix = returns.cov() * 252

# Volatilité du portefeuille

portfolio_volatility = np.sqrt(
    weights.T @ cov_matrix @ weights
)

# Score ESG moyen

portfolio_esg = np.dot(weights, esg_scores)

print(f"Rendement : {portfolio_return:.2%}")
print(f"Volatilité : {portfolio_volatility:.2%}")
print(f"Score ESG : {portfolio_esg:.1f}")

Rendement : 11.88%
Volatilité : 17.49%
Score ESG : 80.8


## portefeuille ESG

Au lieu de pondérer à 20 % partout, on pondère selon le score ESG :

In [4]:
weights_esg = esg_scores / esg_scores.sum()

portfolio_return_esg = np.dot(
    weights_esg,
    annual_return
)

portfolio_vol_esg = np.sqrt(
    weights_esg.T
    @ cov_matrix
    @ weights_esg
)

portfolio_esg_score = np.dot(
    weights_esg,
    esg_scores
)

print("Portefeuille ESG")
print(f"Rendement : {portfolio_return_esg:.2%}")
print(f"Volatilité : {portfolio_vol_esg:.2%}")
print(f"ESG : {portfolio_esg_score:.1f}")

Portefeuille ESG
Rendement : 11.56%
Volatilité : 17.28%
ESG : 81.2


## Analyse du risque

In [7]:
# Sharpe ratio

risk_free_rate = 0.02

sharpe_ratio = (
    portfolio_return - risk_free_rate
) / portfolio_volatility

# Value at Risk (VaR)


# Maximum Drawdown


# Expected Shortfall

## Qualité des données ESG

Analyse de la qualité de données ESG : couverture, cohérence, détection d'anomalies et traitement des données manquantes. (comme dans le cours de ML de kaggle)

In [8]:
# Données manquantes

df.isna().sum()

# Valeurs aberrantes

from scipy.stats import zscore

outliers = np.abs(
    zscore(df["ESG Score"])
) > 3

NameError: name 'df' is not defined

## Optimisation de portefeuille

construire :

*   portefeuille minimum variance
*   portefeuille maximum Sharpe
*   portefeuille ESG contraint

In [ ]:
from scipy.optimize import minimize

## **ESG Portfolio Optimizer: Can Sustainable Portfolios Beat Traditional Portfolios ?**